# ML-03 — Frame Your Lane as an ML Task

# 1. My Lane as an ML Task

## Lane
Ranking Signal Analysis

## Task type
Ranking / scoring

The goal is to score or rank content items according to how likely they
are to experience a decline in search performance, using observable
signals available before the outcome.

This is primarily a ranking/scoring problem because the useful output is
not just a yes/no decision. We want to prioritize pages for investigation
by assigning them a score or ordering them from higher to lower priority.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*


The initial proxy target is whether a content item experiences a significant
decline in search impressions.

A content item will be considered declining when its recent 30-day
impressions are less than 80% of its previous 30-day impressions.

Conceptually:

is_declining = 1 if recent_30_day_impressions <
                    0.8 × previous_30_day_impressions
               else 0

The model would use signals that were available before the outcome window,
such as previous impressions, query diversity, query concentration,
and other observable search-performance signals.

I will avoid outcome-derived variables such as `trend_pct` because they
would leak information about the target.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*


## Primary metric: Precision@K

The primary success metric will be Precision@K, for example Precision@50.

Precision@50 answers:

"Of the 50 pages that the system prioritizes for investigation,
how many actually experience the target decline?"

This metric fits the business action because the team cannot investigate
every page at once. They need a useful shortlist of high-priority pages.

A good model should therefore put a high proportion of genuinely
declining pages near the top of the ranked list.

A secondary evaluation would be client-held-out validation to check
whether the ranking signal generalizes across clients rather than only
working for pages from clients seen during training.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*


## One row = one content item/page for a client

The analysis operates at the content-item level.

Each row represents one content item belonging to a client, with observable
search-performance and query-level signals attached to that content item.

The model therefore makes a page/content-item level prioritization rather
than a client-level or day-level prediction.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["is_declining"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

df[[
    "trend_direction",
    "is_declining"
]].head(10)

,trend_direction,is_declining
0,down,1
1,down,1
2,down,1
3,stable,0
4,down,1
5,down,1
6,down,1
7,stable,0
8,down,1
9,down,1


In [8]:
print(df.columns.tolist())
for i, col in enumerate(df.columns):
    print(i, col)

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
0 content_id
1 client_id
2 search_volume
3 competition
4 competition_level
5 cpc
6 content_type
7 main_intent
8 word_count
9 char_count
10 provider_used
11 model_used
12 impressions_90d
13 clicks_90d
14 pageviews_90d
15 sessions_

In [2]:
import os

print("Current directory:", os.getcwd())
print("\nFiles/folders here:")
print(os.listdir())

Current directory: /content

Files/folders here:
['.config', 'sample_data']


In [3]:
import os
import subprocess

REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

os.chdir(REPO_DIR)

print("Current directory:", os.getcwd())

Current directory: /content/flyrank-ml-internship-starter


In [7]:
df[[
    "impressions_90d",
    "avg_position",
    "ctr",
    "trend_direction"
]].head(10)

,impressions_90d,avg_position,ctr,trend_direction
0,3803,10.6,0.76,down
1,15320,20.3,0.05,down
2,12581,36.5,0.09,down
3,11751,6.2,0.49,stable
4,19140,44.0,0.13,down
5,3970,8.5,0.03,down
6,20,7.0,0.00,down
7,1724,21.2,0.06,stable
8,32574,46.0,0.09,down
9,1240,4.9,0.16,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*


A fixed rule could prioritize pages using one manually chosen condition,
for example:

    if impressions are high and the page is old:
        prioritize the page

This is simple and interpretable, but it requires us to decide the
thresholds and combinations ourselves.

ML can consider multiple signals simultaneously and learn which
combinations are useful for identifying high-priority pages.

For example, a page's previous impressions, query diversity, query
concentration, and other observable signals may interact in ways that
are difficult to capture with one fixed rule.

ML is therefore useful when the goal is to produce a ranked shortlist
from many signals rather than apply one manually selected threshold.

However, ML should only be preferred if it improves the ranking metric
on held-out data. A more complicated model is not automatically better.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.


### ML task type
Ranking / scoring.

### Target / proxy
Whether a content item experiences a significant decline in search
impressions.

### Success metric
Precision@K, with Precision@50 as an initial primary metric.

### Unit of analysis
One content item/page for a client.

### Action supported
Prioritize pages for content/SEO investigation and optimization.

### Why ML
Multiple observable signals can be combined to produce a ranked
priority list instead of relying on a single manually chosen rule.

### Leakage check
Outcome-derived variables such as `trend_pct` must not be used as
predictors because they contain information used to construct the target.

### Main validation concern
The model should be evaluated on held-out data and preferably with
client-level separation to test whether the signal generalizes.